# C

In [1]:
from datasets import load_dataset

# Load only English subset, streaming to avoid RAM explosion
dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)

# Take 1M samples
import itertools
samples = list(itertools.islice(dataset, 1_000_000))
texts = [s['text'] for s in samples]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

In [7]:
import random, re

def inject_errors(sentence):
    """Inject realistic grammar errors into a clean sentence."""
    errors = []
    tokens = sentence.split()
    if len(tokens) < 4:
        return sentence, sentence  # skip short sentences
    
    r = random.random()
    
    if r < 0.25:
        # Subject-verb disagreement: "he go" instead of "he goes"
        # Replace a simple present 3rd-person verb
        sentence_err = re.sub(r'\b(goes|runs|has|is|was)\b', 
                               lambda m: {'goes':'go','runs':'run','has':'have',
                                          'is':'are','was':'were'}[m.group()], 
                               sentence, count=1)
        return sentence_err, sentence
    
    elif r < 0.50:
        # Wrong article: "a" vs "an"
        sentence_err = re.sub(r'\ban\b', 'a', sentence, count=1)
        return sentence_err, sentence
    
    elif r < 0.70:
        # Missing comma in compound sentence
        sentence_err = re.sub(r',\s*(and|but|or)\b', r' \1', sentence, count=1)
        return sentence_err, sentence
    
    elif r < 0.85:
        # Wrong tense: simple past → base form
        sentence_err = re.sub(r'\b(went|ran|told|said|came)\b',
                               lambda m: {'went':'go','ran':'run','told':'tell',
                                          'said':'say','came':'come'}[m.group()],
                               sentence, count=1)
        return sentence_err, sentence
    
    else:
        # Double negation: "not ... nothing"
        sentence_err = re.sub(r"\bdon't\s+have\s+any\b", "don't have no", sentence, count=1)
        return sentence_err, sentence

# Build dataset
pairs = [inject_errors(s) for text in texts 
         for s in re.split(r'(?<=[.!?])\s+', text) 
         if 10 < len(s.split()) < 50]
# Filter: keep only pairs where an error was actually injected
pairs = [(err, clean) for err, clean in pairs if err != clean]

In [8]:
from sklearn.model_selection import train_test_split
train_pairs, test_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
train_pairs, val_pairs = train_test_split(train_pairs, test_size=0.1, random_state=42)

Phase 1: Model 1 — Bag of Words Baseline

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Build classification dataset
X = [err for err, _ in train_pairs] + [clean for _, clean in train_pairs]
y = [0]*len(train_pairs) + [1]*len(train_pairs)  # 0=error, 1=correct

# BoW
vectorizer = CountVectorizer(max_features=20000, ngram_range=(1,2))
X_vec = vectorizer.fit_transform(X)

# Train
clf = LogisticRegression(max_iter=500)
clf.fit(X_vec, y)

# Evaluate on test set
X_test = [err for err, _ in test_pairs] + [clean for _, clean in test_pairs]
y_test = [0]*len(test_pairs) + [1]*len(test_pairs)
X_test_vec = vectorizer.transform(X_test)
preds = clf.predict(X_test_vec)

print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"F1: {f1_score(y_test, preds):.3f}")

Phase 2: Model 2 — LSTM Without Attention

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import  numpy as np
# Tokenize
MAX_VOCAB = 30000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
all_text = [s for pair in train_pairs for s in pair]
tokenizer.fit_on_texts(all_text)

def encode(pairs):
    src = tokenizer.texts_to_sequences([p[0] for p in pairs])
    tgt = tokenizer.texts_to_sequences([p[1] for p in pairs])
    return (pad_sequences(src, maxlen=MAX_LEN, padding='post'),
            pad_sequences(tgt, maxlen=MAX_LEN, padding='post'))

X_train, y_train = encode(train_pairs)
X_val, y_val = encode(val_pairs)

# Encoder
encoder_inputs = tf.keras.Input(shape=(MAX_LEN,))
enc_emb = Embedding(MAX_VOCAB, 128)(encoder_inputs)
_, state_h, state_c = LSTM(256, return_state=True)(enc_emb)

# Decoder — shape must match y[:, :-1] which is MAX_LEN - 1
decoder_inputs = tf.keras.Input(shape=(MAX_LEN - 1,))  # ← fix here
dec_emb = Embedding(MAX_VOCAB, 128)(decoder_inputs)
dec_out = LSTM(256, return_sequences=True)(dec_emb,
                                           initial_state=[state_h, state_c])
outputs = TimeDistributed(Dense(MAX_VOCAB, activation='softmax'))(dec_out)

model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(
    [X_train, y_train[:, :-1]], y_train[:, 1:, np.newaxis],
    validation_data=([X_val, y_val[:, :-1]], y_val[:, 1:, np.newaxis]),
    epochs=10, batch_size=64
)

Phase 3: Model 3 — LSTM With Attention

In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V  = Dense(1)
    
    def call(self, query, values):
        # query: decoder hidden state [batch, 1, hidden]
        # values: all encoder outputs [batch, seq_len, hidden]
        query_exp = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(values) + self.W2(query_exp)))
        weights = tf.nn.softmax(score, axis=1)  # [batch, seq_len, 1]
        context = weights * values
        context = tf.reduce_sum(context, axis=1)  # [batch, hidden]
        return context, weights

# Modified decoder: at each step, compute context from all encoder outputs
# then concatenate context + decoder embedding before LSTM

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_attention(input_sentence, output_sentence, attention_weights):
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(attention_weights, xticklabels=input_sentence.split(),
                yticklabels=output_sentence.split(), cmap='YlOrRd', ax=ax)
    ax.set_xlabel("Input (erroneous)")
    ax.set_ylabel("Output (corrected)")
    plt.title("Attention Weights")
    plt.tight_layout()
    plt.savefig("attention_heatmap.png", dpi=150)

Phase 4: Model 4 — Transformer

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

MODEL_NAME = "t5-small"  # Use t5-base if you have GPU RAM
tokenizer_t5 = T5Tokenizer.from_pretrained(MODEL_NAME)
model_t5 = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

class GECDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self): return len(self.pairs)
    
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_enc = self.tokenizer(
            "grammar: " + src,  # T5 task prefix
            max_length=self.max_len, truncation=True,
            padding='max_length', return_tensors='pt')
        tgt_enc = self.tokenizer(
            tgt, max_length=self.max_len, truncation=True,
            padding='max_length', return_tensors='pt')
        labels = tgt_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100  # ignore padding in loss
        return {
            'input_ids': src_enc['input_ids'].squeeze(),
            'attention_mask': src_enc['attention_mask'].squeeze(),
            'labels': labels
        }

train_dataset = GECDataset(train_pairs[:100_000], tokenizer_t5)  # subset for speed
val_dataset   = GECDataset(val_pairs[:10_000], tokenizer_t5)

training_args = TrainingArguments(
    output_dir='./t5-gec',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    warmup_steps=500,
    fp16=True,  # enable if GPU supports it
    logging_steps=100,
)

trainer = Trainer(model=model_t5, args=training_args,
                  train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
import errant  # pip install errant (GEC-specific metric)

# 1. BLEU score — standard MT metric, measures n-gram overlap
references = [[ref.split()] for _, ref in test_pairs]
hypotheses = [model_predict(src).split() for src, _ in test_pairs]
bleu = corpus_bleu(references, hypotheses)

# 2. Exact match accuracy — did the model produce the exact correct sentence?
exact = sum(h == r[0] for h, r in zip(hypotheses, references)) / len(hypotheses)

# 3. F0.5 score via ERRANT — the standard GEC metric
# Precision-weighted: penalizes over-correction more than missing errors
# pip install errant; python -m spacy download en_core_web_sm

The Final Challenge (Dataset Noise)

In [ ]:
import language_tool_python  # pip install language-tool-python
tool = language_tool_python.LanguageTool('en-US')

noisy_samples = []
for text in texts[:50_000]:  # sample 50k
    matches = tool.check(text)
    if matches:
        noisy_samples.append({
            'text': text[:200],
            'errors': [(m.ruleId, m.message, m.context) for m in matches[:3]]
        })

print(f"Samples with detected grammar issues: {len(noisy_samples)}")